# Evaluation of CIFAR2020 Results

Objective:
Compare locally reproduced α,β-CROWN results against the official VNN-COMP CIFAR2020 benchmark.

Metrics:
- Verification agreement
- Runtime comparison
- Disagreement analysis
- Statistical summary

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:

official = pd.read_csv("../data/official/cifar2020.csv")
local = pd.read_csv("../data/experiments/csv/cifar2020.csv")

In [3]:
official.head()

,benchmark,onnx_path,vnnlib_path,total_time,result,solver_time
0,cifar2020,./benchmarks/cifar2020/nets/cifar10_2_255_simp...,./benchmarks/cifar2020/specs/cifar10/cifar10_s...,6.733640,SAT,6.646123
1,cifar2020,./benchmarks/cifar2020/nets/cifar10_2_255_simp...,./benchmarks/cifar2020/specs/cifar10/cifar10_s...,14.718358,UNSAT,14.609764
2,cifar2020,./benchmarks/cifar2020/nets/cifar10_2_255_simp...,./benchmarks/cifar2020/specs/cifar10/cifar10_s...,30.070079,UNSAT,41.810964
3,cifar2020,./benchmarks/cifar2020/nets/cifar10_2_255_simp...,./benchmarks/cifar2020/specs/cifar10/cifar10_s...,14.656227,UNSAT,14.570861
4,cifar2020,./benchmarks/cifar2020/nets/cifar10_2_255_simp...,./benchmarks/cifar2020/specs/cifar10/cifar10_s...,30.074108,UNSAT,57.078539


In [4]:
local.head()

,benchmark,onnx_path,vnnlib_path,total_time,result,solver_time
0,cifar2020,./benchmarks/cifar2020/nets/cifar10_2_255_simp...,./benchmarks/cifar2020/specs/cifar10/cifar10_s...,0.4304,SAT,0.7916
1,cifar2020,./benchmarks/cifar2020/nets/cifar10_2_255_simp...,./benchmarks/cifar2020/specs/cifar10/cifar10_s...,0.8574,UNSAT,1.2850
2,cifar2020,./benchmarks/cifar2020/nets/cifar10_2_255_simp...,./benchmarks/cifar2020/specs/cifar10/cifar10_s...,0.8194,UNSAT,7.5653
3,cifar2020,./benchmarks/cifar2020/nets/cifar10_2_255_simp...,./benchmarks/cifar2020/specs/cifar10/cifar10_s...,0.7879,UNSAT,0.9444
4,cifar2020,./benchmarks/cifar2020/nets/cifar10_2_255_simp...,./benchmarks/cifar2020/specs/cifar10/cifar10_s...,0.7847,UNSAT,14.6692


In [8]:
assert len(official) == len(local)
assert set(official.onnx_path) == set(local.onnx_path)
assert set(official.vnnlib_path) == set(local.vnnlib_path)


In [13]:
comparison = official.merge(
    local,
    on=["benchmark","onnx_path","vnnlib_path"],
    suffixes=("_official", "_local")
)

In [14]:
comparison.head()

,benchmark,onnx_path,vnnlib_path,total_time_official,result_official,solver_time_official,total_time_local,result_local,solver_time_local
0,cifar2020,./benchmarks/cifar2020/nets/cifar10_2_255_simp...,./benchmarks/cifar2020/specs/cifar10/cifar10_s...,6.733640,SAT,6.646123,0.4304,SAT,0.7916
1,cifar2020,./benchmarks/cifar2020/nets/cifar10_2_255_simp...,./benchmarks/cifar2020/specs/cifar10/cifar10_s...,14.718358,UNSAT,14.609764,0.8574,UNSAT,1.2850
2,cifar2020,./benchmarks/cifar2020/nets/cifar10_2_255_simp...,./benchmarks/cifar2020/specs/cifar10/cifar10_s...,30.070079,UNSAT,41.810964,0.8194,UNSAT,7.5653
3,cifar2020,./benchmarks/cifar2020/nets/cifar10_2_255_simp...,./benchmarks/cifar2020/specs/cifar10/cifar10_s...,14.656227,UNSAT,14.570861,0.7879,UNSAT,0.9444
4,cifar2020,./benchmarks/cifar2020/nets/cifar10_2_255_simp...,./benchmarks/cifar2020/specs/cifar10/cifar10_s...,30.074108,UNSAT,57.078539,0.7847,UNSAT,14.6692


In [15]:
comparison["match"] = (
    comparison.result_official ==
    comparison.result_local
)

In [18]:
instances = len(comparison)
matches = comparison["match"].sum()
mismatches = instances - matches
agreement_pct = matches / instances * 100

print(f"Instances : {instances}")
print(f"Matches   : {matches}")
print(f"Mismatch  : {mismatches}")
print(f"Agreement : {agreement_pct:.0f}%")

Instances : 147
Matches   : 137
Mismatch  : 10
Agreement : 93%


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true = comparison["result_official"]
y_pred = comparison["result_local"]
labels = sorted(set(y_true) | set(y_pred))

cm = confusion_matrix(y_true, y_pred, labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix: official (true) vs local (pred)")
plt.show()

In [19]:
pd.crosstab(
    comparison.result_local,
    comparison.result_official
)

result_official,SAT,UNSAT,timeout
result_local,,,
SAT,34,0,0
UNSAT,0,103,0
unknown,1,0,9
